In [ ]:
# Analytics勉強会のNotebook
# タイタニック号の生還者予測(3)
# モデルを作成する

In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
warnings.simplefilter('ignore')

In [ ]:
# (1)データ確認

In [ ]:
#(1)Kaggle
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
#
train_df

In [ ]:
# データの内容
# PassengerId　識別子
# Survived　生存
# Pclass　旅客等級
# Name　氏名
# Sex　性別
# Age　年齢
# SibSp　兄弟姉妹・配偶者の数
# Parch　親・子供の数
# Ticket　チケット番号
# Fare　料金
# Cabin　客室番号
# Embarked　出発港

In [ ]:
train_df.describe()

In [ ]:
#(1)Kaggle
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')
#
test_df

In [ ]:
test_df.describe()

In [ ]:
# (2)可視化

In [ ]:
Graph_Discribe = train_df.copy()
Graph_Discribe = Graph_Discribe.drop('PassengerId',axis=1)
Graph_Discribe = Graph_Discribe.drop('Name',axis=1)
Graph_Discribe = Graph_Discribe.drop('Ticket',axis=1)
Graph_Discribe = Graph_Discribe.drop('Cabin',axis=1)
Graph_Discribe = Graph_Discribe.drop('Embarked',axis=1)

In [ ]:
plt.figure(figsize=(18,6))
for i, col in enumerate(Graph_Discribe.columns):
    plt.subplot(2,4,i+1)
    plt.hist(Graph_Discribe[col], color='orange')
    plt.title(col)
plt.tight_layout()

In [ ]:
Pclass = train_df['Survived'].groupby(train_df['Pclass']).mean()
plt.bar(Pclass.index,Pclass.values, color='orange')
plt.xticks(Pclass.index)
plt.xlabel('Pclass')
plt.title('Relation between Survived and Pclass')

In [ ]:
Gender = train_df['Survived'].groupby(train_df['Sex']).mean()
plt.bar(Gender.index,Gender.values, color='orange')
plt.xticks(Gender.index)
plt.xlabel('Gender')
plt.title('Relation between Survived and Gender')

In [ ]:
graph1 = sns.FacetGrid(train_df, col='Survived')
graph1.map(plt.hist, 'Age', bins=20, color='orange')

In [ ]:
SibSp = train_df['Survived'].groupby(train_df['SibSp']).mean()
plt.bar(SibSp.index,SibSp.values, color='orange')
plt.xticks(SibSp.index)
plt.xlabel('SibSp')
plt.title('Relation between Survived and SibSp')

In [ ]:
Parch = train_df['Survived'].groupby(train_df['Parch']).mean()
plt.bar(Parch.index,Parch.values, color='orange')
plt.xticks(Parch.index)
plt.xlabel('Parch')
plt.title('Relation between Survived and Parch')

In [ ]:
grid = sns.FacetGrid(train_df, row='Embarked', height=2.2, aspect=1.6)
grid.map(sns.pointplot, 'Pclass', 'Survived', 'Sex', palette='deep')
grid.add_legend()

In [ ]:
# (3)クレンジング

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.tail()

In [ ]:
# Ageは中央値に置き換え
train_df['Age'].fillna(train_df['Age'].median(), inplace=True)

In [ ]:
# Cabinは削除
train_df = train_df.drop('Cabin',axis=1)

In [ ]:
# EmbarkedはCに置き換え
train_df['Embarked'].fillna('C', inplace=True)

In [ ]:
train_df.tail()

In [ ]:
# (4)前処理

In [ ]:
train_df = train_df.drop('PassengerId',axis=1)

In [ ]:
labelencoder = LabelEncoder()

In [ ]:
train_df['Sex'] = labelencoder.fit_transform(train_df['Sex'])
train_df.tail()

In [ ]:
train_df = train_df.drop('Ticket',axis=1)

In [ ]:
Embarked = pd.get_dummies(train_df['Embarked'], drop_first=True)
Embarked.columns = ['Embarked-Q','Embarked-S']

In [ ]:
Pclass = pd.get_dummies(train_df['Pclass'], drop_first=True)
Pclass.columns = ['Pclass2','Pclass3']

In [ ]:
train_df.tail()

In [ ]:
# (5)特徴量エンジニアリング

In [ ]:
# アイデア1：年齢で分類
train_df['Age_Classify'] = pd.qcut(train_df['Age'] ,4)
AgeClassify = train_df['Survived'].groupby(train_df['Age_Classify']).mean()
AgeClassify

In [ ]:
# アイデア2：料金で分類
train_df['Fare_Classify'] = pd.qcut(train_df['Fare'] ,6)
FareClassify = train_df['Survived'].groupby(train_df['Fare_Classify']).mean()
FareClassify

In [ ]:
# アイデア3：家族の人数
train_df['Family_Number'] = train_df['SibSp'] + train_df['Parch']
FamilyNumber = train_df['Survived'].groupby(train_df['Family_Number']).mean()
FamilyNumber

In [ ]:
# アイデア4：単独乗船
train_df['Single'] = np.where(train_df['Family_Number']==0,1,0)
Single = train_df['Survived'].groupby(train_df['Single']).mean()
Single

In [ ]:
# アイデア5：名前の敬称
train_df['Title'] = train_df['Name'].str.extract(r',\s*([^\.]*)\s*\.',expand=False) 
Mr = ['Mr']
Crew1 = ['Don','Rev','Capt']
Crew2 = ['Major','Col','Dr']
Women_Master = ['Mrs','Miss','Master']
Affluence = ['Mme','Ms','Lady','Sir','Mlle','the Countess','Jonkheer']
train_df['Title_Group'] = np.where(train_df['Title'] == Mr[0], 'Mr', 'Affluence')
train_df['Title_Group'] = np.where(train_df['Title'].isin(Crew1), 'Crew1', train_df['Title_Group'])
train_df['Title_Group'] = np.where(train_df['Title'].isin(Crew2), 'Crew2', train_df['Title_Group'])
train_df['Title_Group'] = np.where(train_df['Title'].isin(Women_Master), 'Women_Master', train_df['Title_Group'])
train_df['Title_Group'] = np.where(train_df['Title'].isin(Affluence), 'Affluence', train_df['Title_Group'])
TitlePlot = train_df['Survived'].groupby(train_df['Title_Group']).mean()
TitlePlot

In [ ]:
# 特徴量を追加した人はここにコードを追加
# 
# 

In [ ]:
train_df.head()

In [ ]:
train_df['Age_Classify'] = pd.qcut(train_df['Age'],4,labels=['Age1','Age2','Age3','Age4'])
train_df.head()

In [ ]:
train_df['Fare_Classify'] = pd.qcut(train_df['Fare'],6,labels=['Fare1','Fare2','Fare3','Fare4','Fare5','Fare6'])
train_df.head()

In [ ]:
# 特徴量を追加した人はここにコードを追加
# 
Age = pd.get_dummies(train_df['Age_Classify'], drop_first=True)
Fare = pd.get_dummies(train_df['Fare_Classify'], drop_first=True)
Title_Groups = pd.get_dummies(train_df['Title_Group'], drop_first=True)
train_df = pd.concat([train_df, Pclass, Embarked, Age, Fare, Title_Groups],axis=1)

In [ ]:
# 特徴量を追加した人は下記のコードを修正
#
train_df = train_df.drop(columns=['Pclass','Name','Embarked','Fare','Age','Age_Classify','Fare_Classify','Title','Title_Group'])

In [ ]:
train_df.columns

In [ ]:
train_df.head()

In [ ]:
# Test
y_train = train_df['Survived']
x_train = train_df.drop('Survived',axis=1)

In [ ]:
# Support Vector Machines
Svm = SVC()

In [ ]:
cross_val_score(Svm, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
# (6)モデルを作る

In [ ]:
# ロジスティック回帰
Logistic = LogisticRegression()
Logistic.fit(x_train, y_train)

In [ ]:
cross_val_score(Logistic, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
# ランダムフォレスト
Random = RandomForestClassifier(random_state = 0, oob_score = True)
Random.fit(x_train, y_train)

In [ ]:
cross_val_score(Random, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
# SVM
Svm = SVC()
Svm.fit(x_train, y_train)

In [ ]:
cross_val_score(Svm, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
# 追加したいモデルがあればここに書く
# モデル4

In [ ]:
# 追加したいモデルがあればここに書く
# モデル5

In [ ]:
# (7)テストデータ作成/予測

In [ ]:
# テストデータ作成
test = test_df.copy()

In [ ]:
# テストデータのクレンジング
test.isnull().sum()

In [ ]:
test.isnull().sum()
test['Age'].fillna(test['Age'].median(), inplace=True)
test['Fare'].fillna(test['Fare'].median(), inplace=True)
test = test.drop('Cabin',axis=1)

In [ ]:
# テストデータの前処理

In [ ]:
PassengerId = test['PassengerId']

In [ ]:
test['Sex'] = labelencoder.fit_transform(test_df['Sex'])

In [ ]:
test = test.drop('Ticket',axis=1)
test_Embarked = pd.get_dummies(test['Embarked'], drop_first=True)
test_Embarked.columns = ['Embarked-Q','Embarked-S']
test_Pclass = pd.get_dummies(test['Pclass'], drop_first=True)
test_Pclass.columns = ['Pclass2','Pclass3']

In [ ]:
# テストデータの特徴量エンジニアリング

In [ ]:
# 特徴量を追加した人はコードを追加
# 
test['Age_Classify'] = pd.qcut(test['Age'] ,4)
test['Fare_Classify'] = pd.qcut(test['Fare'] ,6)
test['Family_Number'] = test['SibSp'] + test['Parch']
test['Single'] = np.where(test['Family_Number']==0,1,0)
test['Title'] = test['Name'].str.extract(r',\s*([^\.]*)\s*\.',expand=False) 
test['Title_Group'] = np.where(test['Title'] == Mr[0], 'Mr', 'Affluence')
test['Title_Group'] = np.where(test['Title'].isin(Crew1), 'Crew1', test['Title_Group'])
test['Title_Group'] = np.where(test['Title'].isin(Crew2), 'Crew2', test['Title_Group'])
test['Title_Group'] = np.where(test['Title'].isin(Women_Master), 'Women_Master', test['Title_Group'])
test['Title_Group'] = np.where(test['Title'].isin(Affluence), 'Affluence', test['Title_Group'])

In [ ]:
# 特徴量を追加した人はコードを追加
# 
test['Age_Classify'] = pd.qcut(test['Age'],4,labels=['Age1','Age2','Age3','Age4'])
test['Fare_Classify'] = pd.qcut(test['Fare'],6,labels=['Fare1','Fare2','Fare3','Fare4','Fare5','Fare6'])
test_Age = pd.get_dummies(test['Age_Classify'], drop_first=True)
test_Fare = pd.get_dummies(test['Fare_Classify'], drop_first=True)
test_Title_Groups = pd.get_dummies(test['Title_Group'], drop_first=True)
test = pd.concat([test, test_Pclass, test_Embarked, test_Age, test_Fare, test_Title_Groups],axis=1)

In [ ]:
# 特徴量を追加した人はコードを修正
# 
test = test.drop(columns=['PassengerId', 'Pclass','Name','Embarked','Fare','Age','Age_Classify','Fare_Classify','Title','Title_Group'])

In [ ]:
test.columns

In [ ]:
train_df.columns

In [ ]:
# ロジスティック回帰

In [ ]:
Logistic_pred = Logistic.predict(test)
Logistic_pred

In [ ]:
# ランダムフォレスト

In [ ]:
Random_pred = Random.predict(test)
Random_pred

In [ ]:
# SVM

In [ ]:
Svm_pred = Svm.predict(test)
Svm_pred

In [ ]:
# 前回はここまで
# 今回はここから
# (9)チューニング

In [ ]:
# ロジスティック回帰

In [ ]:
print(Logistic.get_params())

In [ ]:
param_l = {'C':[1e-5,1e-4,1e-3,1e-2,1,1e1,1e2,1e3,1e4,1e5,]}

In [ ]:
GridSearch_l = GridSearchCV(Logistic,param_l,cv=6, scoring='accuracy')

In [ ]:
GridSearch_l.fit(x_train, y_train)
GridSearch_l.best_params_

In [ ]:
cross_val_score(GridSearch_l, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
optimised_l = GridSearch_l.best_estimator_

In [ ]:
print(optimised_l.get_params())

In [ ]:
Logistic_pred_modify = optimised_l.predict(test)
Logistic_pred_modify

In [ ]:
# ランダムフォレスト

In [ ]:
print(Random.get_params())

In [ ]:
param_R = {'criterion' :['gini'],'n_estimators' : [300,350,400],'max_depth':[5,6],'min_samples_leaf': [4,5], 'max_leaf_nodes': [10],'min_impurity_decrease': [0],'max_features' : [1] }

In [ ]:
gridsearch_R = GridSearchCV(Random, param_R, cv=5, scoring='accuracy')

In [ ]:
gridsearch_R.fit(x_train, y_train)
gridsearch_R.best_params_

In [ ]:
cross_val_score(gridsearch_R, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
optimised_R = gridsearch_R.best_estimator_

In [ ]:
print(optimised_R.get_params())

In [ ]:
Random_pred_modify = optimised_R.predict(test)
Random_pred_modify

In [ ]:
# SVM

In [ ]:
print(Svm.get_params())

In [ ]:
param_S = {'degree': [1,2,3,4,5,6,7,8,9,10]}

In [ ]:
gridsearch_S = GridSearchCV(Svm, param_S, cv=5, scoring='accuracy')

In [ ]:
gridsearch_S.fit(x_train, y_train)
gridsearch_S.best_params_

In [ ]:
cross_val_score(gridsearch_S, x_train, y_train, scoring = 'accuracy', cv = 5).mean()

In [ ]:
optimised_S = gridsearch_S.best_estimator_

In [ ]:
print(optimised_S.get_params())

In [ ]:
Svm_pred_modify = optimised_S.predict(test)
Svm_pred_modify

In [ ]:
# モデルを選択する。
#
#(1)チューニングなし
#(1.1)ロジスティック回帰
#result = pd.DataFrame({'PassengerId':PassengerId,'Survived':Logistic_pred})
#(1.2)ランダムフォレスト
#result = pd.DataFrame({'PassengerId':PassengerId,'Survived':Random_pred})
#(1.3)SVM
#result = pd.DataFrame({'PassengerId':PassengerId,'Survived':Svm_pred})
#(1.4)モデルを使した人はここに追記
#
#(1.5)モデルを使した人はここに追記
#
#(2)チューニングあり
#(2.1)ロジスティック回帰+チューニング
#result = pd.DataFrame({'PassengerId':PassengerId,'Survived':Logistic_pred_modify})
#(2.2)ランダムフォレスト+チューニング
#推奨版　
#Kaggle Acu=0.79665, Top 5% Rank
result = pd.DataFrame({'PassengerId':PassengerId,'Survived':Random_pred_modify})
#(2.3)SVM+チューニング
#result = pd.DataFrame({'PassengerId':PassengerId,'Survived':Svm_pred_modify})
#(2.4)モデルを使した人はここに追記
#
#(2.5)モデルを使した人はここに追記
#

In [ ]:
result

In [ ]:
# KaggleにSubmit
result.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
# 練習問題
# パラメータを修正してチューニングしてください。